In [0]:
%run ./engine

In [0]:
# ---- dead OMOP 5.4 event-link columns dropped from training (per source table) ----
_DEAD = {
    "measurement": ["visit_detail_id", "measurement_event_id", "meas_event_field_concept_id"],
    "observation": ["visit_detail_id", "observation_event_id", "obs_event_field_concept_id"],
    "condition_occurrence": ["visit_detail_id"],
    "drug_exposure": ["visit_detail_id"],
    "procedure_occurrence": ["visit_detail_id"],
    "device_exposure": ["visit_detail_id"],
}

# parent-window (visit) for the 6 in-visit events: clamp their date cols into the parent visit's
# [visit_start_date, visit_end_date]. Consumed by the assemble stage.
def _vwin(child_cols):
    return {"parent_start": "visit_start_date", "parent_end": "visit_end_date", "child_cols": child_cols}

_TABLES = {
    # ---- reference (independent subjects; internal FKs assigned post-hoc) ----
    "location": TableSpec(name="location", group="reference", pk="location_id"),
    "care_site": TableSpec(name="care_site", group="reference", pk="care_site_id",
        ref_fks=["location_id"], internal_ref_parent={"location_id": "location"}),
    "provider": TableSpec(name="provider", group="reference", pk="provider_id",
        ref_fks=["care_site_id"], internal_ref_parent={"care_site_id": "care_site"}),

    # ---- core: subject = person ----
    "person": TableSpec(name="person", group="core", pk="person_id",
        ref_fks=["location_id", "provider_id", "care_site_id"],
        date_cols=["birth_datetime"]),
    "observation_period": TableSpec(name="observation_period", group="core",
        pk="observation_period_id", id_type="int", context_fk=("person_id", "person"),
        date_cols=["observation_period_start_date", "observation_period_end_date"]),
    "death": TableSpec(name="death", group="core", pk=None, context_fk=("person_id", "person"),
        base_rate_table=True,
        date_cols=["death_date", "death_datetime"]),
    "condition_era": TableSpec(name="condition_era", group="core", pk="condition_era_id",
        id_type="int", context_fk=("person_id", "person"),
        date_cols=["condition_era_start_date", "condition_era_end_date"]),
    "drug_era": TableSpec(name="drug_era", group="core", pk="drug_era_id", id_type="int",
        context_fk=("person_id", "person"),
        date_cols=["drug_era_start_date", "drug_era_end_date"]),
    "dose_era": TableSpec(name="dose_era", group="core", pk="dose_era_id",
        context_fk=("person_id", "person"),
        date_cols=["dose_era_start_date", "dose_era_end_date"]),
    "observation_person": TableSpec(name="observation_person", group="core", pk="observation_id",
        context_fk=("person_id", "person"), source_table="observation",
        where="visit_occurrence_id IS NULL", split_group="observation",
        drop_cols=_DEAD["observation"],
        date_cols=["observation_date", "observation_datetime"]),
    "visit_occurrence": TableSpec(name="visit_occurrence", group="core", pk="visit_occurrence_id",
        context_fk=("person_id", "person"), ref_fks=["provider_id", "care_site_id"],
        self_fk="preceding_visit_occurrence_id",
        date_cols=["visit_start_date", "visit_start_datetime", "visit_end_date", "visit_end_datetime"]),
    "measurement": TableSpec(name="measurement", group="core", pk="measurement_id",
        context_fk=("visit_occurrence_id", "visit_occurrence"), ref_fks=["provider_id"],
        drop_cols=_DEAD["measurement"],
        caps={"per_parent": 50, "per_subject": 2000},
        parent_window=_vwin(["measurement_date", "measurement_datetime"]),
        date_cols=["measurement_date", "measurement_datetime"]),
    "drug_exposure": TableSpec(name="drug_exposure", group="core", pk="drug_exposure_id",
        context_fk=("visit_occurrence_id", "visit_occurrence"), ref_fks=["provider_id"],
        drop_cols=_DEAD["drug_exposure"], caps={"per_parent": 50},
        parent_window=_vwin(["drug_exposure_start_date", "drug_exposure_start_datetime",
                             "drug_exposure_end_date", "drug_exposure_end_datetime", "verbatim_end_date"]),
        date_cols=["drug_exposure_start_date", "drug_exposure_start_datetime",
                   "drug_exposure_end_date", "drug_exposure_end_datetime", "verbatim_end_date"]),
    "condition_occurrence": TableSpec(name="condition_occurrence", group="core",
        pk="condition_occurrence_id", context_fk=("visit_occurrence_id", "visit_occurrence"),
        ref_fks=["provider_id"], drop_cols=_DEAD["condition_occurrence"],
        parent_window=_vwin(["condition_start_date", "condition_start_datetime",
                             "condition_end_date", "condition_end_datetime"]),
        date_cols=["condition_start_date", "condition_start_datetime",
                   "condition_end_date", "condition_end_datetime"]),
    "procedure_occurrence": TableSpec(name="procedure_occurrence", group="core",
        pk="procedure_occurrence_id", context_fk=("visit_occurrence_id", "visit_occurrence"),
        ref_fks=["provider_id"], drop_cols=_DEAD["procedure_occurrence"], caps={"per_parent": 50},
        parent_window=_vwin(["procedure_date", "procedure_datetime",
                             "procedure_end_date", "procedure_end_datetime"]),
        date_cols=["procedure_date", "procedure_datetime",
                   "procedure_end_date", "procedure_end_datetime"]),
    "device_exposure": TableSpec(name="device_exposure", group="core", pk="device_exposure_id",
        context_fk=("visit_occurrence_id", "visit_occurrence"), ref_fks=["provider_id"],
        drop_cols=_DEAD["device_exposure"], caps={"per_parent": 50},
        parent_window=_vwin(["device_exposure_start_date", "device_exposure_start_datetime",
                             "device_exposure_end_date", "device_exposure_end_datetime"]),
        date_cols=["device_exposure_start_date", "device_exposure_start_datetime",
                   "device_exposure_end_date", "device_exposure_end_datetime"]),
    "observation_visit": TableSpec(name="observation_visit", group="core", pk="observation_id",
        context_fk=("visit_occurrence_id", "visit_occurrence"), ref_fks=["provider_id"],
        source_table="observation", where="visit_occurrence_id IS NOT NULL",
        split_group="observation", drop_cols=_DEAD["observation"],
        parent_window=_vwin(["observation_date", "observation_datetime"]),
        date_cols=["observation_date", "observation_datetime"]),
}

# Today's EXACT hand-ordered topo list -> pin it so the generic engine reproduces it byte-for-byte.
_TABLE_ORDER = [
    "location", "care_site", "provider",
    "person",
    "observation_period", "death", "condition_era", "drug_era", "dose_era",
    "observation_person",
    "visit_occurrence",
    "measurement", "drug_exposure", "condition_occurrence",
    "procedure_occurrence", "device_exposure", "observation_visit",
]

OMOP_CONFIG = DatasetConfig(
    name="omop",
    source={"catalog": "4_prod", "schema": "omop"},
    target={"catalog": "8_dev", "schema": "synth_omop",
            "volume": "/Volumes/8_dev/synth_omop/artifacts"},
    subject="person",
    privacy_unit="person",
    tables=_TABLES,
    table_order=_TABLE_ORDER,
    dp=dict(enabled=True, max_epsilon=2.0, value_protection_epsilon=3.0, delta=1e-6,
            noise_multiplier=1.5, max_grad_norm=1.0),
    profiler=dict(epsilon_profiler=1.0, seqlen_parent_cap=5, cat_per_patient_cap=100,
                  seqlen_bins=[0, 1, 2, 3, 5, 10, 25, 50, 100, 1_000_000]),
    rare=dict(seqlen_support_min=0.05, rate_tol=0.02, privacy_safe_support=20, min_expected=5),
    disable_valprot_tables=["drug_era", "dose_era", "drug_exposure",
                            "procedure_occurrence", "device_exposure"],
    valprot_override={"death": True},
    reference_chain=["provider", "care_site", "location"],
    lifespan_rule={"birth_table": "person", "birth_col": "birth_datetime",
                   "death_table": "death", "death_col": "death_datetime"},
    # PII (nulled at write time + hard-gated 100% NULL). R4a: provider/care_site identifiers
    # added (they were previously trained as freetext_topk categoricals -> verbatim source names).
    pii={"location": ["address_1", "address_2", "zip", "location_source_value",
                      "latitude", "longitude", "LSOA"],
         "provider": ["provider_name", "npi", "dea"],
         "care_site": ["care_site_name"]},
    # freetext_topk: unbounded free-text modelled as top-K categorical. R4a: provider_name / npi /
    # dea / care_site_name REMOVED (now PII). Only device/drug free-text remains.
    freetext_topk=["sig", "lot_number", "unique_device_id", "production_id"],
    pk_band_size=100_000_000,
    model_limits={"max_training_time": 120, "max_epochs": 50},
    enable_model_report=False,
    extra=dict(
        src_persons=3_480_621,
        cohort=dict(persons=30_000, meas_cap=2_000, obs_cap=2_000, per_visit_keep=50,
                    ref_pool=20_000, care_site_all=True, seed=42),
        output=dict(persons=100_000),
        max_pandas_rows=15_000_000, max_pandas_gb=12.0,
        spark_date_min="1900-01-01", spark_date_max="2099-12-31",
        observation_slices=["observation_person", "observation_visit"],
        sdtypes_table="6_mgmt.default.sdtypes",
        sdtypes_prefix="omop_",
        null_fk_rate={"measurement": 4190 / 725_342_629},   # measured source visit-null rate
        validate=dict(
            # exact gate/key labels included so the generic validate stage reproduces
            # validation.json verbatim per dataset
            null_rate_check=dict(name="obs_visit_null_rate (source ~0.072)",
                                 table="observation", col="visit_occurrence_id",
                                 lo=0.04, hi=0.20),
            fidelity_ratios=dict(visits_per_person="visit_occurrence",
                                 measurements_per_person="measurement",
                                 drugs_per_person="drug_exposure",
                                 conditions_per_person="condition_occurrence",
                                 death_fraction="death"),
            source_reference={"visits_per_person": 11.54, "measurements_per_person": 208.39,
                              "drugs_per_person": 13.54, "conditions_per_person": 8.57,
                              "death_fraction": 0.041,
                              "note": "measurements_per_person intentionally lower: per-person cap (~p98) truncates the heavy-utiliser tail."},
            cooccurrence=dict(key="within_visit_top_measurements_for_top_visit_type",
                              parent="visit_occurrence", parent_concept="visit_concept_id",
                              child="measurement", child_concept="measurement_concept_id"),
        ),
    ),
)

print(f"omop_config: OMOP_CONFIG built — {len(OMOP_CONFIG.tables)} tables, "
      f"subject={OMOP_CONFIG.subject}, pii tables={sorted(OMOP_CONFIG.pii)}, "
      f"errors={validate_config(OMOP_CONFIG)}")